# Notebook 07 — Gradio Demo

**Features:**
- Toggle RAG on/off
- Toggle between base and fine-tuned model
- Top-K slider
- Retrieved chunks display with source + score
- "Compare 4 configs" tab
- 6 example questions
- `demo.launch(share=True)` for 72-hour public Colab URL

> **Prerequisites:** Notebooks 01–04 must be complete.

In [1]:
from pathlib import Path
import os

# Tự động tìm đường dẫn
if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = Path("../")

print(f'Base directory: {BASE}')
print(os.listdir(BASE))

Mounted at /content/drive
Base directory: /content/drive/MyDrive/NLP_Final/Source
['.gitignore', 'requirements.txt', 'scripts', '.git', 'data', 'notebooks', 'models', 'results', '.env']


In [2]:
%%capture
!pip install -q trl==0.11.4 \
    transformers==4.45.0 peft==0.13.0 accelerate==1.0.1
!pip install -q -U bitsandbytes
!pip install -q "numpy<2" faiss-cpu==1.8.0 sentence-transformers==3.2.1
!pip uninstall gradio gradio_client -y
!pip install gradio==3.50.2
print('Packages installed.')

## 1. Load All Components

In [3]:
import os, torch, pickle
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    AutoModelForSequenceClassification,
)
from peft import PeftModel
from typing import List, Dict, Optional
import json
from datetime import datetime

MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
SFT_PATH   = os.path.join(BASE, "models", "sft_checkpoint")
PPO_PATH   = os.path.join(BASE, "models", "ppo_checkpoint")
REWARD_PATH = os.path.join(BASE, "models", "reward_model")
INDEX_DIR  = os.path.join(BASE, "data", "vector_store", "faiss_index")

SYSTEM_PROMPT = (
    "Bạn là trợ lý AI của Trường Đại học Tôn Đức Thắng (TDTU). "
    "Bạn trả lời các câu hỏi về quy chế, quy định, chính sách của trường một cách chính xác và đầy đủ. "
    "Trả lời bằng tiếng Việt. Nếu không có đủ thông tin, hãy nói rõ điều đó.\n\n"
    "HƯỚNG DẪN:\n"
    "- Nếu câu hỏi có từ 'thế', 'vậy', 'còn' tham chiếu đến câu trước, hãy nhớ ngữ cảnh\n"
    "- Giữ thái độ thân thiện, chuyên nghiệp\n"
    "- Trả lời chính xác dựa trên quy chế TDTU"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_DTYPE = (
    torch.bfloat16 if device == "cuda" and torch.cuda.get_device_properties(0).total_memory > 30e9
    else torch.float16 if device == "cuda"
    else torch.float32
)
print(f"Device: {device}  |  dtype: {COMPUTE_DTYPE}")

# --- FAISS ---
print("Loading FAISS index...")
faiss_index = faiss.read_index(os.path.join(INDEX_DIR, "index.faiss"))
with open(os.path.join(INDEX_DIR, "index.pkl"), "rb") as f:
    index_chunks = pickle.load(f)

# --- Embedder ---
print("Loading embedding model...")
embedder = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder", device="cuda")

# --- Tokenizer & base/sft models ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    llm_int8_enable_fp32_cpu_offload=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(SFT_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)
base_model.eval()

print("Loading fine-tuned model (SFT adapter)...")
ft_model = PeftModel.from_pretrained(base_model, SFT_PATH, adapter_name="sft")
ft_model.eval()

# --- PPO adapter (optional) ---
RLHF_AVAILABLE = False
if os.path.exists(PPO_PATH):
    print("Loading PPO adapter (Config E)...")
    ft_model.load_adapter(PPO_PATH, adapter_name="ppo")
    ft_model.set_adapter("sft")
    RLHF_AVAILABLE = True
    print("PPO adapter loaded.")
else:
    print(f"PPO checkpoint not found at {PPO_PATH}")

# --- Reward model (optional) ---
reward_model = None
reward_tokenizer = None
if os.path.exists(REWARD_PATH):
    print("Loading reward model...")
    reward_tokenizer = AutoTokenizer.from_pretrained(REWARD_PATH, trust_remote_code=True)
    _reward_base = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=1,
        quantization_config=bnb_config, device_map="auto", trust_remote_code=True,
    )
    reward_model = PeftModel.from_pretrained(_reward_base, REWARD_PATH)
    reward_model.eval()
    print("Reward model loaded.")
else:
    print(f"Reward model not found at {REWARD_PATH}")

REWARD_AVAILABLE = reward_model is not None
print("All components loaded.")


# ---------------------------------------------------------------------------
# TDTU Chatbot với Memory
# ---------------------------------------------------------------------------

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
class TDTUChatbotWithMemory:
    def __init__(self, model, tokenizer, reward_model=None, reward_tokenizer=None):
        self.model = model
        self.tokenizer = tokenizer
        self.reward_model = reward_model
        self.reward_tokenizer = reward_tokenizer
        self.conversation_history = []  # Lưu lịch sử hội thoại
        self.max_history_turns = 8      # Giữ tối đa 8 lượt (4 cặp QA)
        self.current_context_chunks = []

    def reset(self):
        """Reset conversation history"""
        self.conversation_history = []
        self.current_context_chunks = []
        return "✅ Đã xóa lịch sử hội thoại"

    def get_history_summary(self) -> str:
        """Get summary of conversation history"""
        if not self.conversation_history:
            return "Chưa có hội thoại nào"

        summary = []
        for i, turn in enumerate(self.conversation_history[-6:]):
            role = "👤 User" if turn["role"] == "user" else "🤖 Assistant"
            content = turn["content"][:100] + "..." if len(turn["content"]) > 100 else turn["content"]
            summary.append(f"{role}: {content}")
        return "\n".join(summary)

    def save_conversation(self, filepath: str = None):
        """Save conversation to JSON file"""
        if filepath is None:
            filepath = f"conversation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        # Thêm metadata
        save_data = {
            "timestamp": datetime.now().isoformat(),
            "num_turns": len(self.conversation_history) // 2,
            "history": self.conversation_history
        }

        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(save_data, f, ensure_ascii=False, indent=2)
        return f"💾 Đã lưu hội thoại vào {filepath}"

    def load_conversation(self, filepath: str):
        """Load conversation from JSON file"""
        with open(filepath, 'r', encoding='utf-8') as f:
            load_data = json.load(f)

        if "history" in load_data:
            self.conversation_history = load_data["history"]
        else:
            self.conversation_history = load_data

        return f"📂 Đã tải {len(self.conversation_history)//2} lượt hội thoại"

    def build_prompt_with_history(self, question: str, context_chunks: List[str] = None) -> str:
        """Build prompt with conversation history and optional RAG context"""
        prompt = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'

        # Add RAG context if available
        if context_chunks:
            context_str = "\n\n".join(f"[{i+1}] {chunk}" for i, chunk in enumerate(context_chunks[:3]))
            prompt += f'<|im_start|>system\nTHÔNG TIN THAM KHẢO:\n{context_str}<|im_end|>\n'

        # Add conversation history (chỉ lấy gần nhất)
        for turn in self.conversation_history[-self.max_history_turns*2:]:
            prompt += f'<|im_start|>{turn["role"]}\n{turn["content"]}<|im_end|>\n'

        # Add current question
        prompt += f'<|im_start|>user\n{question}<|im_end|>\n'
        prompt += f'<|im_start|>assistant\n'

        return prompt

    def retrieve_with_history_context(self, question: str, k: int = 5) -> List[str]:
        """Retrieve chunks, considering conversation history for better retrieval"""
        # Tạo query mở rộng từ lịch sử nếu có
        enhanced_query = question

        if len(self.conversation_history) >= 2:
            # Lấy câu hỏi trước đó để hiểu ngữ cảnh
            last_user_turn = None
            for turn in reversed(self.conversation_history):
                if turn["role"] == "user":
                    last_user_turn = turn["content"]
                    break

            if last_user_turn and any(word in question.lower() for word in ["thế", "vậy", "còn", "cũng"]):
                # Kết hợp câu hỏi trước với câu hỏi hiện tại
                enhanced_query = f"{last_user_turn} {question}"
                print(f"🔍 Enhanced query: {enhanced_query[:100]}...")

        # Retrieve chunks
        q_vec = embedder.encode([enhanced_query], normalize_embeddings=True).astype("float32")
        scores, indices = faiss_index.search(q_vec, k)

        chunks = []
        for j, i in enumerate(indices[0]):
            if i < len(index_chunks):
                chunks.append(index_chunks[i]["text"])

        self.current_context_chunks = chunks
        return chunks

    def chat(self,
             question: str,
             use_rag: bool = True,
             use_finetuned: bool = True,
             top_k: int = 5,
             max_new_tokens: int = 512) -> tuple[str, str]:
        """Main chat function with memory"""
        if not question.strip():
            return "Vui lòng nhập câu hỏi.", ""

        # Chọn model
        model = ft_model if use_finetuned else base_model

        # Retrieve context (nếu bật RAG)
        context_chunks = []
        context_display = ""
        if use_rag:
            context_chunks = self.retrieve_with_history_context(question, k=top_k)
            if context_chunks:
                context_display = "\n\n".join(
                    f"**📄 Chunk {i+1}:** {chunk[:300]}{'...' if len(chunk) > 300 else ''}"
                    for i, chunk in enumerate(context_chunks[:3])
                )
            else:
                context_display = "*(Không tìm thấy tài liệu liên quan)*"

        # Build prompt với history
        prompt = self.build_prompt_with_history(question, context_chunks if use_rag else None)

        # Generate response
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.7 if use_rag else 0.3,
                top_p=0.9,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=IM_END_ID,
            )

        generated = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(generated, skip_special_tokens=True).strip()

        # Clean response
        if '<|im_end|>' in response:
            response = response.split('<|im_end|>')[0]

        # Lưu vào history
        self.conversation_history.append({"role": "user", "content": question})
        self.conversation_history.append({"role": "assistant", "content": response})

        # Giới hạn history
        if len(self.conversation_history) > self.max_history_turns * 2:
            self.conversation_history = self.conversation_history[-self.max_history_turns * 2:]

        return response, context_display

    def get_reward_score(self, text: str) -> float:
        """Get reward score for evaluation"""
        if not self.reward_model:
            return float("nan")
        inputs = self.reward_tokenizer(text, return_tensors="pt", truncation=True,
                                      max_length=512).to(self.reward_model.device)
        with torch.no_grad():
            return self.reward_model(**inputs).logits.squeeze().item()


In [ ]:
print(next(ft_model.parameters()).device)  # Phải là cuda:0
# Nếu là cpu, bạn đã sai. Move model lên GPU:
ft_model = ft_model.to('cuda')
base_model = base_model.to('cuda')

In [ ]:
# ---------------------------------------------------------------------------
# Gradio Interface - Phiên bản hoàn chỉnh không lỗi
# ---------------------------------------------------------------------------

import gradio as gr

# Khởi tạo chatbot với memory
chatbot_memory = TDTUChatbotWithMemory(ft_model, tokenizer, reward_model, reward_tokenizer)

# Sample questions
EXAMPLES_VI = [
    "Sinh viên bị đình chỉ học tập trong trường hợp nào?",
    "Điều kiện để được xét học bổng khuyến khích học tập là gì?",
    "Quy định về trang phục của sinh viên trong trường như thế nào?",
    "Sinh viên có thể chuyển ngành học không? Điều kiện và thủ tục ra sao?",
    "Mức xử lý kỷ luật khi sinh viên gian lận thi cử là gì?",
    "Điều kiện tốt nghiệp đại học tại TDTU là gì?",
]

# Test multi-turn examples
MULTI_TURN_EXAMPLES = [
    ("Học bổng TDTU có những loại nào?", "Vậy loại khuyến khích học tập cần GPA bao nhiêu?"),
    ("Thủ tục nhập học gồm những gì?", "Cần mang ảnh thẻ không?"),
    ("Quy định về việc bảo lưu kết quả ra sao?", "Thời gian tối đa được bảo lưu là bao lâu?"),
]


def chat_with_memory(message, history, use_rag, use_finetuned, top_k):
    """Gradio chat function with memory - đúng thứ tự arguments"""
    if not message or not message.strip():
        return history

    response, context = chatbot_memory.chat(
        question=message,
        use_rag=use_rag,
        use_finetuned=use_finetuned,
        top_k=int(top_k),
        max_new_tokens=512
    )

    # Update history
    if history is None:
        history = []
    history.append((message, response))

    return history


def respond(message, history):
    """Simple response function cho tab chat chính"""
    if not message or not message.strip():
        return history

    response, _ = chatbot_memory.chat(
        question=message,
        use_rag=True,
        use_finetuned=True,
        top_k=5
    )

    if history is None:
        history = []
    history.append((message, response))

    return history


def reset_conversation():
    """Reset conversation"""
    chatbot_memory.reset()
    return []


def save_conversation_fn():
    """Save conversation"""
    return chatbot_memory.save_conversation()


def compare_configs(question):
    """Compare different configurations"""
    if not question or not question.strip():
        return "Vui lòng nhập câu hỏi.", "Vui lòng nhập câu hỏi.", "Vui lòng nhập câu hỏi.", "Vui lòng nhập câu hỏi.", ""

    original_model = chatbot_memory.model

    # Config A: base + no RAG
    chatbot_memory.model = base_model
    ans_a, _ = chatbot_memory.chat(question, use_rag=False, use_finetuned=False, max_new_tokens=256)

    # Config B: base + RAG
    chatbot_memory.reset()
    chatbot_memory.model = base_model
    ans_b, _ = chatbot_memory.chat(question, use_rag=True, use_finetuned=False, max_new_tokens=256)

    # Config C: fine-tuned + no RAG
    chatbot_memory.reset()
    chatbot_memory.model = ft_model
    ans_c, _ = chatbot_memory.chat(question, use_rag=False, use_finetuned=True, max_new_tokens=256)

    # Config D: fine-tuned + RAG
    chatbot_memory.reset()
    chatbot_memory.model = ft_model
    ans_d, context_d = chatbot_memory.chat(question, use_rag=True, use_finetuned=True, max_new_tokens=256)

    # Khôi phục model gốc
    chatbot_memory.model = original_model
    chatbot_memory.reset()

    return ans_a, ans_b, ans_c, ans_d, context_d

# ---------------------------------------------------------------------------
# Gradio Interface
# ---------------------------------------------------------------------------

with gr.Blocks(title="TDTU QA System", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🎓 Hệ thống Hỏi-Đáp Quy chế TDTU

        **Tính năng:**
        - 🧠 **Nhớ ngữ cảnh** - Chatbot nhớ những gì bạn đã hỏi trước đó
        - 🔄 **Multi-turn conversation** - Hỏi tiếp theo tự nhiên
        - 📚 **RAG tích hợp** - Tra cứu từ tài liệu quy chế
        """
    )

    with gr.Tab("💬 Chat"):
        chatbot_interface = gr.Chatbot(label="Hội thoại", height=500)
        msg = gr.Textbox(label="Câu hỏi của bạn", placeholder="Nhập câu hỏi ở đây...", lines=2)

        with gr.Row():
            submit_btn = gr.Button("📤 Gửi", variant="primary")
            reset_btn = gr.Button("🗑️ Xóa lịch sử", variant="secondary")

        gr.Examples(examples=EXAMPLES_VI, inputs=msg, label="📌 Câu hỏi mẫu")

        submit_btn.click(respond, [msg, chatbot_interface], chatbot_interface).then(
            lambda: "", None, msg
        )
        msg.submit(respond, [msg, chatbot_interface], chatbot_interface).then(
            lambda: "", None, msg
        )
        reset_btn.click(reset_conversation, None, chatbot_interface)

    with gr.Tab("📊 So sánh cấu hình"):
        gr.Markdown("### So sánh 4 cấu hình model")
        compare_q = gr.Textbox(label="Câu hỏi để so sánh", lines=2)
        compare_btn = gr.Button("🔄 So sánh", variant="primary")

        with gr.Row():
            with gr.Column():
                gr.Markdown("**Config A:** Base, no RAG")
                ans_a = gr.Textbox(label="", lines=8, interactive=False)
                gr.Markdown("**Config C:** Fine-tuned, no RAG")
                ans_c = gr.Textbox(label="", lines=8, interactive=False)
            with gr.Column():
                gr.Markdown("**Config B:** Base + RAG")
                ans_b = gr.Textbox(label="", lines=8, interactive=False)
                gr.Markdown("**Config D:** Fine-tuned + RAG ★")
                ans_d = gr.Textbox(label="", lines=8, interactive=False)

        compare_context = gr.Markdown("### 📚 Context từ RAG\n*Chưa có*")

        compare_btn.click(
            fn=compare_configs,
            inputs=[compare_q],
            outputs=[ans_a, ans_b, ans_c, ans_d, compare_context]
        )

        gr.Examples(examples=EXAMPLES_VI[:3], inputs=compare_q, label="Câu hỏi mẫu")

    with gr.Tab("ℹ️ Thông tin"):
        gr.Markdown(f"""
        ### Chi tiết mô hình

        | Thành phần | Chi tiết |
        |---|---|
        | Base LLM | Qwen/Qwen2.5-3B-Instruct |
        | Fine-tuning | QLoRA (r=16, alpha=32) |
        | Embedding | Vietnamese-bi-encoder |
        | Memory | 8 turns (4 cặp QA) |
        | RLHF | {'✅ Có' if RLHF_AVAILABLE else '❌ Không'} |

        ### 💡 Mẹo sử dụng

        1. Hỏi tiếp theo bằng các từ: "thế", "vậy", "còn"
        2. Ví dụ: "Học bổng TDTU có những loại nào?" → "Vậy loại khuyến khích cần GPA bao nhiêu?"
        3. Bấm **Xóa lịch sử** để bắt đầu chủ đề mới
        """)

# Launch
print("🚀 Đang khởi động TDTU Chatbot...")
demo.launch(share=True)